# MB25 | Fase 4 | Consultas SQL do Datalake

**Tabela:** `dev_procurement.corp_curated.vw_ds_log_mb25`  
**Recorte:** centro `4128`  
**Chave:** `num_reserva + num_item_reserva`

Este notebook contém as seis consultas SQL acordadas para coletar as evidências do lado Datalake.

**Execução:** rode todas as células no Databricks e exporte o resultado de cada consulta em CSV, mantendo os nomes sugeridos nos títulos.

## 1. Volumetria e chave

**Nome sugerido do arquivo:** `01_volumetria_chave.csv`

In [0]:
%sql
    SELECT
        COUNT(*) AS total_linhas,
        COUNT(DISTINCT num_reserva) AS reservas,
        COUNT(DISTINCT CONCAT_WS('|',
            COALESCE(CAST(num_reserva AS STRING), '<NULL>'),
            COALESCE(CAST(num_item_reserva AS STRING), '<NULL>')
        )) AS chaves
    FROM dev_procurement.corp_curated.vw_ds_log_mb25
    WHERE CAST(cod_centro AS STRING) = '4128';

## 2. Duplicidades da chave

**Nome sugerido do arquivo:** `02_duplicidades_chave.csv`

In [0]:
%sql
    SELECT
        num_reserva,
        num_item_reserva,
        COUNT(*) AS qtd
    FROM dev_procurement.corp_curated.vw_ds_log_mb25
    WHERE CAST(cod_centro AS STRING) = '4128'
    GROUP BY
        num_reserva,
        num_item_reserva
    HAVING COUNT(*) > 1
    ORDER BY
        qtd DESC,
        num_reserva,
        num_item_reserva;

## 3. Campos suspeitos identificados na exploração

**Nome sugerido do arquivo:** `03_campos_suspeitos.csv`

In [0]:
%sql
    SELECT
        COUNT(*) AS total,
        SUM(CASE
            WHEN desc_material IS NULL
              OR LOWER(TRIM(CAST(desc_material AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')
            THEN 1 ELSE 0
        END) AS desc_material_vazia,
        SUM(CASE
            WHEN dt_reserva IS NULL
              OR LOWER(TRIM(CAST(dt_reserva AS STRING))) IN ('', '00000000', 'null', 'none', 'nan', '#', '-')
            THEN 1 ELSE 0
        END) AS dt_reserva_invalida,
        SUM(CASE
            WHEN cod_deposito IS NULL
              OR LOWER(TRIM(CAST(cod_deposito AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')
            THEN 1 ELSE 0
        END) AS deposito_vazio,
        SUM(CASE
            WHEN nm_usuario IS NULL
              OR LOWER(TRIM(CAST(nm_usuario AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')
            THEN 1 ELSE 0
        END) AS usuario_vazio,
        SUM(CASE WHEN UPPER(TRIM(CAST(ind_item_eliminado AS STRING))) = 'X' THEN 1 ELSE 0 END) AS itens_eliminados,
        SUM(CASE WHEN UPPER(TRIM(CAST(ind_registro_final AS STRING))) = 'X' THEN 1 ELSE 0 END) AS registros_finais
    FROM dev_procurement.corp_curated.vw_ds_log_mb25
    WHERE CAST(cod_centro AS STRING) = '4128';

## 4. Qualidade e intervalo das datas

**Nome sugerido do arquivo:** `04_qualidade_datas.csv`

In [0]:
%sql
    SELECT
        MIN(CASE
            WHEN TRIM(CAST(dt_necessidade AS STRING)) RLIKE '^[0-9]{8}$'
             AND TRIM(CAST(dt_necessidade AS STRING)) <> '00000000'
            THEN TO_DATE(TRIM(CAST(dt_necessidade AS STRING)), 'yyyyMMdd')
        END) AS min_dt_necessidade_valida,
        MAX(CASE
            WHEN TRIM(CAST(dt_necessidade AS STRING)) RLIKE '^[0-9]{8}$'
             AND TRIM(CAST(dt_necessidade AS STRING)) <> '00000000'
            THEN TO_DATE(TRIM(CAST(dt_necessidade AS STRING)), 'yyyyMMdd')
        END) AS max_dt_necessidade_valida,
        SUM(CASE
            WHEN dt_necessidade IS NULL
              OR TRIM(CAST(dt_necessidade AS STRING)) IN ('', '00000000')
            THEN 1 ELSE 0
        END) AS dt_necessidade_vazia_ou_zerada,
        SUM(CASE
            WHEN dt_necessidade IS NOT NULL
             AND TRIM(CAST(dt_necessidade AS STRING)) NOT IN ('', '00000000')
             AND TO_DATE(TRIM(CAST(dt_necessidade AS STRING)), 'yyyyMMdd') IS NULL
            THEN 1 ELSE 0
        END) AS dt_necessidade_invalida,

        MIN(CASE
            WHEN TRIM(CAST(dt_reserva AS STRING)) RLIKE '^[0-9]{8}$'
             AND TRIM(CAST(dt_reserva AS STRING)) <> '00000000'
            THEN TO_DATE(TRIM(CAST(dt_reserva AS STRING)), 'yyyyMMdd')
        END) AS min_dt_reserva_valida,
        MAX(CASE
            WHEN TRIM(CAST(dt_reserva AS STRING)) RLIKE '^[0-9]{8}$'
             AND TRIM(CAST(dt_reserva AS STRING)) <> '00000000'
            THEN TO_DATE(TRIM(CAST(dt_reserva AS STRING)), 'yyyyMMdd')
        END) AS max_dt_reserva_valida,
        SUM(CASE
            WHEN dt_reserva IS NULL
              OR TRIM(CAST(dt_reserva AS STRING)) IN ('', '00000000')
            THEN 1 ELSE 0
        END) AS dt_reserva_vazia_ou_zerada,
        SUM(CASE
            WHEN dt_reserva IS NOT NULL
             AND TRIM(CAST(dt_reserva AS STRING)) NOT IN ('', '00000000')
             AND TO_DATE(TRIM(CAST(dt_reserva AS STRING)), 'yyyyMMdd') IS NULL
            THEN 1 ELSE 0
        END) AS dt_reserva_invalida,

        MIN(CASE
            WHEN TRIM(CAST(dt_criacao AS STRING)) RLIKE '^[0-9]{8}$'
             AND TRIM(CAST(dt_criacao AS STRING)) <> '00000000'
            THEN TO_DATE(TRIM(CAST(dt_criacao AS STRING)), 'yyyyMMdd')
        END) AS min_dt_criacao_valida,
        MAX(CASE
            WHEN TRIM(CAST(dt_criacao AS STRING)) RLIKE '^[0-9]{8}$'
             AND TRIM(CAST(dt_criacao AS STRING)) <> '00000000'
            THEN TO_DATE(TRIM(CAST(dt_criacao AS STRING)), 'yyyyMMdd')
        END) AS max_dt_criacao_valida,
        SUM(CASE
            WHEN dt_criacao IS NULL
              OR TRIM(CAST(dt_criacao AS STRING)) IN ('', '00000000')
            THEN 1 ELSE 0
        END) AS dt_criacao_vazia_ou_zerada,
        SUM(CASE
            WHEN dt_criacao IS NOT NULL
             AND TRIM(CAST(dt_criacao AS STRING)) NOT IN ('', '00000000')
             AND TO_DATE(TRIM(CAST(dt_criacao AS STRING)), 'yyyyMMdd') IS NULL
            THEN 1 ELSE 0
        END) AS dt_criacao_invalida
    FROM dev_procurement.corp_curated.vw_ds_log_mb25
    WHERE CAST(cod_centro AS STRING) = '4128';

## 5. Validação da regra de saldo

A tolerância aplicada é `0,005`.

**Nome sugerido do arquivo:** `05_validacao_saldo.csv`

In [0]:
%sql
    SELECT
        COUNT(*) AS total,
        SUM(CASE
            WHEN qt_necessaria IS NULL
              OR qt_retirada IS NULL
              OR qt_saldo_reserva IS NULL
            THEN 1 ELSE 0
        END) AS nao_validavel_por_nulo,
        SUM(CASE
            WHEN qt_necessaria IS NOT NULL
             AND qt_retirada IS NOT NULL
             AND qt_saldo_reserva IS NOT NULL
             AND ABS(qt_saldo_reserva - (qt_necessaria - qt_retirada)) <= 0.005
            THEN 1 ELSE 0
        END) AS dentro_tolerancia,
        SUM(CASE
            WHEN qt_necessaria IS NOT NULL
             AND qt_retirada IS NOT NULL
             AND qt_saldo_reserva IS NOT NULL
             AND ABS(qt_saldo_reserva - (qt_necessaria - qt_retirada)) > 0.005
            THEN 1 ELSE 0
        END) AS fora_tolerancia,
        MAX(CASE
            WHEN qt_necessaria IS NOT NULL
             AND qt_retirada IS NOT NULL
             AND qt_saldo_reserva IS NOT NULL
            THEN ABS(qt_saldo_reserva - (qt_necessaria - qt_retirada))
        END) AS maior_diferenca
    FROM dev_procurement.corp_curated.vw_ds_log_mb25
    WHERE CAST(cod_centro AS STRING) = '4128';

## 6. Domínio dos indicadores

**Nome sugerido do arquivo:** `06_indicadores.csv`

In [0]:
%sql
    SELECT
        COALESCE(NULLIF(TRIM(CAST(ind_item_eliminado AS STRING)), ''), '<VAZIO>') AS ind_item_eliminado,
        COALESCE(NULLIF(TRIM(CAST(ind_registro_final AS STRING)), ''), '<VAZIO>') AS ind_registro_final,
        COUNT(*) AS qtd
    FROM dev_procurement.corp_curated.vw_ds_log_mb25
    WHERE CAST(cod_centro AS STRING) = '4128'
    GROUP BY
        COALESCE(NULLIF(TRIM(CAST(ind_item_eliminado AS STRING)), ''), '<VAZIO>'),
        COALESCE(NULLIF(TRIM(CAST(ind_registro_final AS STRING)), ''), '<VAZIO>')
    ORDER BY qtd DESC;

## Encerramento

Após executar, exporte os seis resultados em CSV. Para a comparação linha a linha, exporte também separadamente o resultado bruto abaixo usando o editor SQL, o recurso de download da tabela ou outra opção disponível no seu ambiente:

```sql
SELECT *
FROM dev_procurement.corp_curated.vw_ds_log_mb25
WHERE CAST(cod_centro AS STRING) = '4128';
```